# 九州工業大学大学院生命体工学研究科 インターンシップ — 第1部 前半
## グラフ畳み込みを「手で」計算する

### 第1部前半の目的

グラフニューラルネットワーク(GNN)の中心にあるのは**グラフ畳み込み**という演算である。
ライブラリを使えば `GCNConv(...)` と一行書くだけで動いてしまうが、
それでは中で何が起きているのか分からないまま実験が終わってしまう。

そこで第1部前半では、**PyTorch Geometric を一切使わずに、NumPy だけで畳み込みを手計算する**。
そして最後に、自分の手計算の結果が `GCNConv` の出力と**数値まで一致する**ことを確認する。

一致が確認できれば、`GCNConv` はもうブラックボックスではない。
以降のすべての実験を「なぜそうなるのか」で語れるようになる。

### 到達目標
1. 隣接行列 $A$ から対称正規化行列 $\hat{A}$ を自分で構成できる
2. $H' = \hat{A}XW^{\top} + b$ を計算し、`GCNConv` と一致させられる
3. 「1層の畳み込み = 1ホップ先までの情報の混合」であることを確認できる

In [ ]:
import os
import numpy as np
import torch
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_dense_adj

# 実行するたびに同じ結果になるよう乱数を固定する
torch.manual_seed(0)

# 本実習で扱うモデルは小さいため、計算スレッドを増やすとかえって遅くなる
# (並列化のオーバーヘッドが計算量を上回るため。実測で1.4〜1.5倍の差が出る)。
# 1スレッドに固定して実行時間を短縮する。
torch.set_num_threads(1)

print("torch           :", torch.__version__)
print("numpy           :", np.__version__)
import torch_geometric
print("torch_geometric :", torch_geometric.__version__)

---
## 1. 題材のグラフ — Zachary の空手クラブ

ある大学の空手クラブの34人の人間関係を表したグラフである。
辺は「クラブの外でも交流があった」ことを表す。

各頂点には、このグラフを Girvan-Newman 法で分割して得られる
**4つのコミュニティ**のどれに属するかがラベルとして与えられている
（空手クラブは実際には2つに分裂したことで有名だが、
PyTorch Geometric が同梱しているラベルは分裂後の2派閥ではなく4コミュニティである。混同しないこと）。

小さいので**行列を目で見て確かめられる**。手計算の題材として最適である。

In [ ]:
dataset = KarateClub()
data = dataset[0]

N = data.num_nodes
print(f"頂点数       : {N}")
print(f"辺数(有向表現): {data.num_edges}")
print(f"特徴量の形   : {tuple(data.x.shape)}")
print(f"クラス数     : {int(data.y.max()) + 1}")
print()
print("edge_index の形:", tuple(data.edge_index.shape))
print("最初の5本の辺 (始点, 終点):")
for k in range(5):
    print("   ", int(data.edge_index[0, k]), "->", int(data.edge_index[1, k]))

`edge_index` は **2行×辺数列** の行列で、1行目が始点、2行目が終点である。
無向グラフなので、辺 $\{u,v\}$ は $(u \to v)$ と $(v \to u)$ の2本として格納されている。

特徴量 `data.x` は $34\times34$ の**単位行列**である。
つまり「頂点 $i$ の特徴ベクトルは $i$ 番目の成分だけが1」という、
頂点を区別するためだけの特徴量である。
これは後で非常に便利になる（理由は第4節で分かる）。

In [ ]:
X = data.x.numpy()
print("X は単位行列か:", np.allclose(X, np.eye(N)))
print(X[:5, :8])

---
## 2. 隣接行列 $A$ を作る

`edge_index` から隣接行列 $A$ を作る。$A_{ij}=1$ は「頂点 $i$ と $j$ が隣接する」ことを表す。

> **課題1a-1** 次のセルの空欄を埋めて、隣接行列 `A` を完成させよ。

In [ ]:
A = np.zeros((N, N))

src = data.edge_index[0].numpy()   # 始点の配列
dst = data.edge_index[1].numpy()   # 終点の配列

# ----- ここから: A の該当成分を 1 にする -----
# ヒント: NumPy は A[行の配列, 列の配列] = 値 とまとめて代入できる
A[     ,      ] =
# ----- ここまで -----

print("A の形:", A.shape)
print("A の成分の総和:", A.sum(), " (= 有向表現での辺数)")

In [ ]:
# --- 検算 ---
# 性質のチェックだけでは不十分である。
# 例えば A が全てゼロの行列でも「対称」「成分は0か1」「対角は0」を満たしてしまう。
# そこで PyG の関数で作った隣接行列と全要素を比較する。
A_expected = to_dense_adj(data.edge_index, max_num_nodes=N)[0].numpy()
assert A.shape == (N, N), "A の形が違う"
assert np.allclose(A, A_expected), \
    "A が edge_index と一致しない。代入する行・列の順序を確認すること"

# 以下は A の性質の確認 (上の比較が通れば自動的に成り立つ)
assert np.allclose(A, A.T), "無向グラフなので A は対称行列になるはず"
assert set(np.unique(A)) <= {0.0, 1.0}, "A の成分は 0 か 1 のはず"
assert np.allclose(np.diag(A), 0), "自己ループはまだ無いはず"

deg = A.sum(axis=1)
print("検算OK (PyG の隣接行列と全要素一致)")
print(f"次数の最小 {int(deg.min())} / 最大 {int(deg.max())} / 平均 {deg.mean():.2f}")
print("頂点0の隣接頂点:", np.nonzero(A[0])[0].tolist())

---
## 3. 対称正規化行列 $\hat{A}$ を作る

グラフ畳み込みは「自分と隣人の特徴を混ぜる」演算である。ただし素朴に足し合わせると
**次数の大きい頂点ほど値が大きくなってしまう**（この弊害は第1部後半で実測する）。
そこで次の2段階を踏む。

**(a) 自己ループを足す** — 混合のときに自分自身の特徴も残すため
$$\tilde{A} = A + I$$

**(b) 対称正規化する** — 次数による大小を打ち消すため
$$\hat{A} = \tilde{D}^{-1/2}\,\tilde{A}\,\tilde{D}^{-1/2},
\qquad \tilde{D}_{ii} = \sum_{j}\tilde{A}_{ij}$$

成分で書くと、頂点 $i,j$ が隣接するとき
$$\hat{A}_{ij} = \frac{1}{\sqrt{\tilde{d}_i}\sqrt{\tilde{d}_j}}$$
である。**両端の次数が大きい辺ほど重みが小さくなる**ことを式から読み取ってほしい。

> **課題1a-2** 次のセルの空欄を埋めて $\hat{A}$ を完成させよ。

In [ ]:
# ----- ここから -----
A_tilde = A +                              # (a) 自己ループを足す
deg_tilde =                                # 自己ループ込みの次数 (A_tilde の行和)
d_inv_sqrt =                               # 次数の -1/2 乗
# (b) 対称正規化。d_inv_sqrt[:, None] は列ベクトル、d_inv_sqrt[None, :] は行ベクトル
A_hat =
# ----- ここまで -----

print("A_hat の形:", A_hat.shape)
print("A_hat[0, 0] =", A_hat[0, 0])

In [ ]:
# --- 検算 ---
assert np.allclose(A_hat, A_hat.T), "A_hat も対称行列になるはず"
# 非ゼロの位置は A_tilde と完全に一致するはず (辺を消したり増やしたりしていないか)
assert np.array_equal(A_hat != 0, (A + np.eye(N)) != 0), \
    "A_hat の非ゼロ位置が A+I と違う。自己ループの足し忘れか、正規化の掛け方を確認すること"
# 全ての辺について重みが 1/sqrt(d_i d_j) になっているか
_r, _c = np.nonzero(A + np.eye(N))
assert np.allclose(A_hat[_r, _c], 1.0 / np.sqrt(deg_tilde[_r] * deg_tilde[_c])), \
    "A_hat の重みが 1/sqrt(次数の積) になっていない"

i, j = 0, 1
assert A[i, j] == 1, "頂点0と1は隣接しているはず"
expected = 1.0 / np.sqrt(deg_tilde[i] * deg_tilde[j])
assert np.isclose(A_hat[i, j], expected), "A_hat[0,1] の値が式と合わない"

print("検算OK")
print(f"頂点0の次数(自己ループ込み) = {deg_tilde[0]:.0f}")
print(f"頂点1の次数(自己ループ込み) = {deg_tilde[1]:.0f}")
print(f"A_hat[0,1] = {A_hat[0,1]:.6f}  (= 1/sqrt({deg_tilde[0]:.0f}*{deg_tilde[1]:.0f}))")
print()
print("辺ごとに『両端の次数の積』と『A_hat の重み』を対応させる:")
print("(頂点単位の平均ではなく辺単位で見ること。重みは両端の次数で決まるため)")
print()

iu = np.triu_indices(N, k=1)          # 上三角 = 無向辺を1本ずつ数える
is_edge = A[iu] > 0
prod = (deg_tilde[iu[0]] * deg_tilde[iu[1]])[is_edge]   # 両端の次数の積
wgt = A_hat[iu][is_edge]                                # 対応する重み
e_u, e_v = iu[0][is_edge], iu[1][is_edge]
order = np.argsort(prod)

print(f"  {'辺':>10} | {'両端の次数':>12} | {'次数の積':>8} | {'重み':>8}")
print("  " + "-" * 48)
for k in list(order[:3]) + list(order[-3:]):
    print(f"  {e_u[k]:3d} -- {e_v[k]:3d} | "
          f"{deg_tilde[e_u[k]]:5.0f} , {deg_tilde[e_v[k]]:3.0f} | "
          f"{prod[k]:8.0f} | {wgt[k]:8.4f}")

print()
print("重みが 1/sqrt(次数の積) と完全に一致するか:",
      np.allclose(wgt, 1.0 / np.sqrt(prod)))
print(f"次数の積 と 重み の相関係数: {np.corrcoef(prod, wgt)[0,1]:+.4f}"
      "   (負 = 次数が大きいほど重みが小さい)")

---
## 4. 畳み込み1層を手で計算し、`GCNConv` と一致させる

グラフ畳み込み層 `GCNConv` が行う計算は、実は次の一行に尽きる。

$$H' = \hat{A}\,X\,W^{\top} + b$$

- $X$ : 入力の特徴行列（頂点数 × 入力次元）
- $W$ : 重み行列（出力次元 × 入力次元）。PyTorch の `Linear` の流儀で転置が付く
- $b$ : バイアス
- $\hat{A}$ : 先ほど作った正規化隣接行列

$\hat{A}$ を左から掛けることが「隣人と混ぜる」、$W^\top$ を右から掛けることが
「特徴を変換する」に対応している。**これだけ**である。

まず `GCNConv` を1つ作り、その中の $W$ と $b$ を取り出す。

In [ ]:
OUT_DIM = 4
conv = GCNConv(in_channels=data.num_features, out_channels=OUT_DIM)

W = conv.lin.weight.detach().numpy()    # 形: (出力次元, 入力次元)
b = conv.bias.detach().numpy()          # 形: (出力次元,)

print("W の形:", W.shape)
print("b の形:", b.shape)

> **課題1a-3** 次のセルの空欄を埋めて、$H' = \hat{A}XW^{\top} + b$ を手で計算せよ。

In [ ]:
# ----- ここから: NumPy だけで1層分の畳み込みを計算する -----
# ヒント: 行列積は @ 演算子、転置は .T
H_hand =
# ----- ここまで -----

print("H_hand の形:", H_hand.shape)
print(H_hand[:3])

In [ ]:
# --- PyTorch Geometric の GCNConv に同じ入力を通す ---
H_pyg = conv(data.x, data.edge_index).detach().numpy()

diff = np.abs(H_hand - H_pyg).max()
print(f"手計算と GCNConv の最大絶対誤差: {diff:.3e}")
print()
if diff < 1e-5:
    print("一致した。GCNConv がやっていたのは A_hat @ X @ W.T + b だけである。")
else:
    print("一致していない。A_hat の作り方か行列積の順序を見直すこと。")

assert diff < 1e-5, "手計算と GCNConv が一致していない"

---
## 5. 「1層 = 1ホップ」を確かめる

畳み込みを1回行うと、各頂点の新しい特徴は**自分と隣接頂点**の情報だけから作られる。
2回行えば**2ホップ先**まで届く。これがGNNの層数の意味である。

ここで特徴量 $X$ が単位行列であることが効いてくる。
$X=I$ なら $\hat{A}XW^\top = \hat{A}W^\top$ となり、
**頂点 $i$ の出力がどの頂点の入力に依存するかは、$\hat{A}$ の第 $i$ 行の非ゼロ位置そのもの**になる。

> **課題1a-4** 頂点0の1層出力が依存する頂点の集合を求め、
> それが「頂点0の隣接頂点 $\cup$ {頂点0自身}」と一致することを確かめよ。

In [ ]:
v = 0

# ----- ここから: A_hat の第 v 行の非ゼロ成分の位置を求める -----
# ヒント: np.nonzero(...) は非ゼロ成分の添字を返す
dep_1hop = set(                            )
# ----- ここまで -----

neighbors = set(np.nonzero(A[v])[0].tolist())

print(f"頂点{v}の1層出力が依存する頂点数: {len(dep_1hop)}")
print(f"頂点{v}の隣接頂点数            : {len(neighbors)}")
print(f"一致するか: {dep_1hop == neighbors | {v}}")

assert dep_1hop == neighbors | {v}, \
    "1層の依存先が「隣接頂点と自分」に一致しない。A_hat の行の取り方を確認すること"

In [ ]:
# --- 2層にすると2ホップに広がることを確かめる ---
A_hat2 = A_hat @ A_hat
dep_2hop = set(np.nonzero(np.abs(A_hat2[v]) > 1e-12)[0].tolist())

# グラフをたどって2ホップ近傍を列挙する
two_hop = set()
for u in neighbors | {v}:
    two_hop |= set(np.nonzero(A[u])[0].tolist()) | {u}

print(f"頂点{v}の2層出力が依存する頂点数: {len(dep_2hop)}")
print(f"グラフ上の2ホップ近傍の頂点数  : {len(two_hop)}")
print(f"一致するか: {dep_2hop == two_hop}")

assert dep_2hop == two_hop, "2層の依存先が2ホップ近傍に一致しない"
print()

# 層数を増やすと到達範囲がどう広がるか
print("層数 | 到達する頂点数")
print("-" * 22)
M = np.eye(N)
for L in range(1, 7):
    M = M @ A_hat
    reach = int((np.abs(M[v]) > 1e-12).sum())
    print(f"{L:>4} | {reach:>10} / {N}")

**観察**: 層を重ねるほど1つの頂点が参照する範囲が広がり、やがてグラフ全体に届く。

「ならば層を深くするほど情報が増えて精度が上がるはずだ」と考えたくなる。
**それが正しいかどうかを第2部前半で実測する。** 予想を立てておくこと。

---
## 6. 第1部前半の課題

提出はこのノートブック(.ipynb)を実行済みの状態で行うこと。

- **課題1a-1** 隣接行列 $A$ の構成（第2節）
- **課題1a-2** 対称正規化行列 $\hat{A}$ の構成（第3節）
- **課題1a-3** $H'=\hat{A}XW^\top+b$ の手計算と `GCNConv` との一致確認（第4節）
- **課題1a-4** 「1層 = 1ホップ」の確認（第5節）
- **課題1a-5** 次のセルに答えを記述せよ。

### 課題1a-5（記述）

以下に答えを書くこと。

**(1)** $\hat{A}$ を作るとき、なぜ $A$ ではなく $\tilde{A}=A+I$ を正規化したのか。
$I$ を足さずに正規化すると、頂点自身の特徴は出力にどう反映されるか。

（ここに記述）

**(2)** $\hat{A}_{ij}=1/\sqrt{\tilde{d}_i \tilde{d}_j}$ より、次数の大きい頂点どうしを結ぶ辺の重みは小さくなる。
この設計にはどんな意図があると考えられるか。第3節の出力を根拠に述べよ。

（ここに記述）

**(3)** 第5節の表より、層数を増やすと参照範囲が広がることが分かった。
では層数を増やせば精度は上がると予想するか。理由とともに述べよ。
（第2部前半でこの予想を検証する。外れていてもよい。根拠のある予想を書くこと。）

（ここに記述）